In [1]:
library(dplyr)
library(data.table)
library(Matrix)
library(Seurat)
library(ggplot2)
library(SeuratObject)

source('/data/work/liver_hcc_spatial/xxx/Tools/LoadBGI_Spatial_AGA_fix(1).R')


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


Attaching SeuratObject



In [ ]:
samples <- read.csv('/data/work/liver_hcc_spatial/segmentation_samples.csv')
inputdir <- '/data/work/liver_hcc_spatial/03_segmentation/'
outdir_base <- '/data/work/liver_hcc_spatial/04_seurat/'
umi.filter <- 50

colorlist <- c("#FFFF00","#1CE6FF","#FF34FF","#FF4A46","#008941",
               "#006FA6","#A30059","#FFE4E1","#0000A6","#63FFAC",
               "#B79762","#004D43","#8FB0FF","#997D87","#5A0007",
               "#809693","#1B4400","#4FC601","#3B5DFF","#FF2F80",
               "#BA0900","#6B7900","#00C2A0","#FFAA92","#FF90C9",
               "#B903AA","#DDEFFF","#7B4F4B","#A1C299","#0AA6D8",
               "#00A087FF","#4DBBD5FF","#E64B35FF","#3C5488FF","#F38400",
               "#A1CAF1","#C2B280","#848482","#E68FAC","#0067A5",
               "#F99379","#604E97","#F6A600","#B3446C","#DCD300",
               "#882D17","#8DB600","#654522","#E25822","#2B3D26")

for (i in 1:nrow(samples)) {
  id <- samples$id[i]
  cat('\n========== 开始处理:', id, '==========\n')
  
  outdir <- file.path(outdir_base, id)
  if (!dir.exists(outdir)) dir.create(outdir, recursive = TRUE)
  
  setwd(inputdir)
  
  # 创建Seurat对象
  obj <- LoadBGI_Spatial(paste0(id, '_scgem.csv.gz'),
                         outdir = getwd(),
                         bin_data = F,
                         bin_size = 50,
                         cell_mask = T,
                         area_mask = F,
                         save_as = "rds",
                         pro_name = id,
                         UMI_GreyScale_Image = F,
                         assay = "Spatial",
                         slice = id,
                         delete_bg = T,
                         csv = F)
  
  # QC过滤
  obj <- obj[, obj$nCount_Spatial > umi.filter]
  
  # SCTransform聚类
  DefaultAssay(obj) <- 'Spatial'
  obj <- SCTransform(obj, verbose = FALSE, assay = 'Spatial')
  obj <- RunPCA(obj, assay = "SCT", verbose = FALSE)
  obj <- FindNeighbors(obj, reduction = "pca", dims = 1:30)
  obj <- FindClusters(obj, verbose = FALSE)
  obj <- RunUMAP(obj, reduction = "pca", dims = 1:30)
  
  # 保存rds
  saveRDS(obj, paste0(outdir, '/', id, '.SCT.rds'))
  
  # UMAP图
  p <- DimPlot(obj, label = T, reduction = "umap", cols = colorlist)
  ggsave(plot = p, file = paste0(outdir, '/', id, '.UMAP.SCT.png'), width = 8, height = 8)
  
  # Spatial图
  p <- SpatialDimPlot(obj, label = TRUE, label.size = 1, pt.size.factor = 5, stroke = 0)
  ggsave(plot = p, file = paste0(outdir, '/', id, '.Spatial.SCT.png'), width = 3.5, height = 3.5)
  
  # metadata
  meta <- as.data.frame(obj@meta.data)
  write.csv(meta, paste0(outdir, '/', id, '.Seurat.metadata.SCT.csv'), quote = F)
  
  # markers
  DefaultAssay(obj) <- 'SCT'
  markers <- FindAllMarkers(obj, only.pos = T)
  write.csv(markers, paste0(outdir, '/', id, '.markers.SCT.csv'), quote = F)
  
  # cell id
  write.csv(Cells(obj), paste0(outdir, '/', id, '.SCT.cellid.csv'), quote = F)
  
  # 转h5ad
  library(reticulate)
  use_python("/opt/conda/bin/python")
  genes <- as.data.frame(rownames(obj), row.names = rownames(obj))
  names(genes) <- "Gene"
  cells <- as.data.frame(colnames(obj), row.names = colnames(obj))
  names(cells) <- "CellID"
  row_coord <- obj@images[[1]]@coordinates$row
  col_coord <- obj@images[[1]]@coordinates$col
  coordinates <- list(matrix(c(row_coord, col_coord), ncol = 2))
  names(coordinates) <- "spatial"
  ann <- import("anndata")
  ad <- ann$AnnData(X = obj@assays$SCT@data, obs = genes, var = cells,
                    varm = coordinates, layers = list(counts = obj@assays$SCT@counts))
  ad <- ad$T
  ad$write_h5ad(file.path(outdir, paste0(id, "_SCT.h5ad")))
  
  cat('========== 完成:', id, '==========\n')
}

cat('\n全部样本处理完成！\n')